In [ ]:
from qiskit import *
import torch
import numpy as np
import matplotlib.pyplot as plt
import quimb as qu

# from EGATE import *
from NNVQE_HEA import *

import random

seed = 0

torch.manual_seed(seed)
np.random.seed(seed)
random.seed(seed)

In [ ]:

n = 8
depth = 2

In [ ]:

#  [t.tolist() for t in edge_data[1].values()]
cirq, idx = HEA({"params": np.zeros(1000),  "edges": None, "edge_data":None}, n, depth, param_num=True)
print("The number of parameters is", idx)
# print(cirq.depth())
# cirq.draw("mpl")

In [ ]:
def mse(vector1, vector2):
    return np.mean((np.array(vector1) - np.array(vector2)) ** 2)


In [ ]:

def fidelity_torch(psi, phi, eps=1e-12):
    # 동일 shape·dtype 체크 생략 가능
    # psi = psi.detach().numpy()
    # arr = np.asarray(q.full()).ravel()           # shape (d,), 복소 ndarray
    # return torch.as_tensor(arr, dtype=dtype, device=device)
    psi = torch.from_numpy(psi)
    # phi = phi.detach().numpy()
    # print(psi)
    # print(phi)
    psi = psi / torch.linalg.vector_norm(psi)  # 자동으로 복소·실수 모두 처리
    phi = phi / torch.linalg.vector_norm(phi)

    return torch.abs(torch.vdot(psi, phi))**2 

def calculate_dataset_fidelities(states_A, states_B):
    assert len(states_A) == len(states_B), "Datasets must have the same number of states."
    
    fidelities = [fidelity_torch(psi, phi) for psi, phi in zip(states_A, states_B)]

    return fidelities



In [ ]:
def test(n, d, test_edge_full_data, num):
  test_energy = []
  final_state = []
  params = np.random.uniform(0, 2*np.pi, size=idx)
  for j in range(num):
    edge_data = [t.tolist() for t in test_edge_full_data[j].values()]
    edges = list(test_edge_full_data[j].keys())
    
    
    qc = HEA({"params": params, "edges": edges, "edge_data": edge_data}, n, d)
    energy, state = Energy(qc, edges, edge_data, compute_state=True)

    test_energy.append(energy.item())
    final_state.append(torch.tensor(state, dtype=torch.complex128))
  return test_energy, final_state

In [ ]:
test_edge_full_data = torch.load('edge_full_data_test.pt')
# test_latent_data = torch.load('latent_data_test.pt')
analytical_energies_test = torch.load('analytical_energies_test.pt')
analytical_states_test = torch.load('analytical_states_test.pt')

print("analytical_energies_test = ",analytical_energies_test)


In [ ]:
test_energy, final_state = test(n, depth, test_edge_full_data, len(test_edge_full_data))
# relative error

fidelity = calculate_dataset_fidelities(analytical_states_test, final_state)

fidelities = []

for i in range(len(fidelity)):
    fidelities.append(fidelity[i].item())
# print(fidelities)

fidelities = np.array(fidelities)
# print(fidelities)
# relative error
favg = np.mean(fidelities)
plt.plot(
    list(range(len(test_edge_full_data))), fidelities, "-", color="b",linewidth= 1
)
# plt.xlabel("Delta", fontsize=14)
plt.ylabel("Fidelity", fontsize=14)
plt.axhline(favg,linestyle='--', linewidth= 1, label = f"mean = {favg:.5f}",  color = 'red')
# plt.axvspan(-3.0, 3.0, color="darkgrey", alpha=0.5)  # training set span
plt.legend()
plt.show()

In [ ]:
# relative error
err = [a - b for a, b in zip(test_energy, analytical_energies_test)]/ np.abs(analytical_energies_test)
avg = np.mean(err)
plt.plot(
    list(np.linspace(-10.,10.,1000)), err, "-", color="b",linewidth= 1
)
plt.xlabel(r"$\lambda$", fontsize=14)
plt.ylabel("GS Relative Error", fontsize=14)
plt.axhline(avg,linestyle='--', linewidth= 1, label = f"mean = {avg:.5f}",  color = 'red')
plt.axvspan(-3.0, 3.0, color="darkgrey", alpha=0.5)  # training set span
plt.legend()
plt.show()

In [ ]:
mse_value = mse(test_energy, analytical_energies_test)
print([mse_value.item(), avg.item(), favg.item()])

In [ ]:
print("GAE_NNVQE_test_energies_2 = ",test_energy)